In [ ]:
# ============================================================
# SIMILARITY FRAMEWORK V2
# ============================================================
#
# PURPOSE
# -------
# Test whether the conserved international SHAP architecture
# is robust to multiple mathematically distinct ways of
# comparing normalized attribution profiles.
#
# METHODS
# -------
# 1. Cosine similarity
# 2. Aitchison distance (compositional geometry)
# 3. Hellinger distance
# 4. Pairwise Spearman rank correlation
# 5. Kendall's W global rank concordance
#
# For every outcome:
#   A. Full 14-predictor architecture
#   B. BELONG removed -> remaining 13 predictors renormalized
#
# For every metric:
#   - observed cross-system agreement
#   - 10,000 predictor-label permutation nulls
#
# IMPORTANT
# ---------
# NO MODEL FITTING.
# Reads frozen OOF SHAP attribution only.
#
# ============================================================


from pathlib import Path
import math
import time

import numpy as np
import pandas as pd

from scipy.spatial.distance import (
    pdist,
    squareform,
)

from scipy.stats import (
    rankdata,
    spearmanr,
)


# ============================================================
# 1. Configuration
# ============================================================

RANKING_PATH = Path(
    "../results/primary_attribution/"
    "oof_shap_rankings.csv"
)


OUT_DIR = Path(
    "../results/similarity_framework_v2"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FEATURES_14 = [
    "ADAPTIVITY",
    "AGE",
    "ATTLNACT",
    "BELONG",
    "DISCLIMA",
    "EDUSHORT",
    "ESCS",
    "JOYREAD",
    "MASTGOAL",
    "METASPAM",
    "METASUM",
    "ST004D01T",
    "STIMREAD",
    "UNDREM",
]


FEATURES_13 = [
    feature
    for feature in FEATURES_14
    if feature != "BELONG"
]


OUTCOMES = [
    "BEINGBULLIED",
    "BULLY_VERBAL",
    "BULLY_THREAT",
    "BULLY_PHYSICAL",
    "BULLY_RELATIONAL",
]


N_PERMUTATIONS = 10_000

GLOBAL_RANDOM_SEED = 42

BATCH_SIZE = 100

SUM_TOLERANCE = 1e-6


# ============================================================
# 2. Load frozen SHAP results
# ============================================================

rankings = pd.read_csv(
    RANKING_PATH
)


REQUIRED_COLUMNS = {
    "Country",
    "Outcome",
    "Feature",
    "ImportanceShare",
}


missing_columns = (
    REQUIRED_COLUMNS
    -
    set(
        rankings.columns
    )
)


if missing_columns:

    raise RuntimeError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )


rankings[
    "ImportanceShare"
] = pd.to_numeric(
    rankings[
        "ImportanceShare"
    ],
    errors="coerce",
)


# ============================================================
# 3. PROFILE AUDIT
# ============================================================

print("\n")
print("=" * 115)
print("SHAP PROFILE AUDIT")
print("=" * 115)


# ------------------------------------------------------------
# 3A. Duplicate rows
# ------------------------------------------------------------

duplicate_mask = (
    rankings.duplicated(
        subset=[
            "Country",
            "Outcome",
            "Feature",
        ],
        keep=False,
    )
)


n_duplicates = int(
    duplicate_mask.sum()
)


print(
    "Duplicate Country/Outcome/Feature rows:",
    n_duplicates
)


if n_duplicates > 0:

    print(
        rankings.loc[
            duplicate_mask
        ]
        .sort_values(
            [
                "Outcome",
                "Country",
                "Feature",
            ]
        )
        .head(50)
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Duplicate attribution rows found."
    )


# ------------------------------------------------------------
# 3B. Invalid shares
# ------------------------------------------------------------

invalid_share = (
    rankings[
        "ImportanceShare"
    ].isna()
    |
    ~np.isfinite(
        rankings[
            "ImportanceShare"
        ]
    )
    |
    (
        rankings[
            "ImportanceShare"
        ]
        <
        0
    )
)


print(
    "Invalid/non-finite/negative shares:",
    int(
        invalid_share.sum()
    )
)


if invalid_share.any():

    raise RuntimeError(
        "Invalid SHAP importance shares found."
    )


# ------------------------------------------------------------
# 3C. Check exact feature universe
# ------------------------------------------------------------

profile_audit_rows = []


for outcome in OUTCOMES:

    temp = rankings[
        rankings[
            "Outcome"
        ]
        ==
        outcome
    ]


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    for country in systems:

        sub = temp[
            temp[
                "Country"
            ]
            ==
            country
        ]


        observed_features = set(
            sub[
                "Feature"
            ]
        )


        missing_features = (
            set(
                FEATURES_14
            )
            -
            observed_features
        )


        extra_features = (
            observed_features
            -
            set(
                FEATURES_14
            )
        )


        share_sum = float(
            sub[
                "ImportanceShare"
            ]
            .sum()
        )


        profile_audit_rows.append(
            {
                "Outcome":
                    outcome,

                "Country":
                    country,

                "N_features":
                    len(
                        sub
                    ),

                "ShareSum":
                    share_sum,

                "ShareSumError":
                    abs(
                        share_sum
                        -
                        1.0
                    ),

                "MissingFeatures":
                    ";".join(
                        sorted(
                            missing_features
                        )
                    ),

                "ExtraFeatures":
                    ";".join(
                        sorted(
                            extra_features
                        )
                    ),

                "ExactZeros":
                    int(
                        (
                            sub[
                                "ImportanceShare"
                            ]
                            ==
                            0
                        )
                        .sum()
                    ),

                "Below1e8":
                    int(
                        (
                            sub[
                                "ImportanceShare"
                            ]
                            <
                            1e-8
                        )
                        .sum()
                    ),

                "Below1e6":
                    int(
                        (
                            sub[
                                "ImportanceShare"
                            ]
                            <
                            1e-6
                        )
                        .sum()
                    ),

                "Below1e4":
                    int(
                        (
                            sub[
                                "ImportanceShare"
                            ]
                            <
                            1e-4
                        )
                        .sum()
                    ),
            }
        )


profile_audit = pd.DataFrame(
    profile_audit_rows
)


profile_audit.to_csv(
    OUT_DIR
    / "profile_audit.csv",
    index=False,
)


bad_feature_count = (
    profile_audit[
        "N_features"
    ]
    !=
    14
)


bad_sum = (
    profile_audit[
        "ShareSumError"
    ]
    >
    SUM_TOLERANCE
)


bad_feature_identity = (
    (
        profile_audit[
            "MissingFeatures"
        ]
        !=
        ""
    )
    |
    (
        profile_audit[
            "ExtraFeatures"
        ]
        !=
        ""
    )
)


print(
    "Profiles:",
    len(
        profile_audit
    )
)

print(
    "Profiles with !=14 predictors:",
    int(
        bad_feature_count.sum()
    )
)

print(
    "Profiles with share-sum error > tolerance:",
    int(
        bad_sum.sum()
    )
)

print(
    "Profiles with missing/extra predictors:",
    int(
        bad_feature_identity.sum()
    )
)


if (
    bad_feature_count.any()
    or
    bad_sum.any()
    or
    bad_feature_identity.any()
):

    raise RuntimeError(
        "SHAP profile QA failed."
    )


# ------------------------------------------------------------
# 3D. Zero / near-zero audit
# ------------------------------------------------------------

shares = (
    rankings[
        "ImportanceShare"
    ]
    .to_numpy(
        dtype=float
    )
)


positive_shares = shares[
    shares > 0
]


EXACT_ZERO_COUNT = int(
    (
        shares
        ==
        0
    )
    .sum()
)


BELOW_1E8_COUNT = int(
    (
        shares
        <
        1e-8
    )
    .sum()
)


BELOW_1E6_COUNT = int(
    (
        shares
        <
        1e-6
    )
    .sum()
)


BELOW_1E4_COUNT = int(
    (
        shares
        <
        1e-4
    )
    .sum()
)


GLOBAL_MIN_POSITIVE = float(
    positive_shares.min()
)


print("\nZERO / NEAR-ZERO SUMMARY")

print(
    "Exact zero shares:",
    EXACT_ZERO_COUNT
)

print(
    "Shares < 1e-8:",
    BELOW_1E8_COUNT
)

print(
    "Shares < 1e-6:",
    BELOW_1E6_COUNT
)

print(
    "Shares < 1e-4:",
    BELOW_1E4_COUNT
)

print(
    "Minimum positive share:",
    GLOBAL_MIN_POSITIVE
)


zero_rows = (
    rankings[
        rankings[
            "ImportanceShare"
        ]
        ==
        0
    ]
    .copy()
)


zero_rows.to_csv(
    OUT_DIR
    / "exact_zero_shares.csv",
    index=False,
)


if len(
    zero_rows
) > 0:

    print("\nEXACT ZERO SHARES")

    print(
        zero_rows[
            [
                "Country",
                "Outcome",
                "Feature",
                "ImportanceShare",
            ]
        ]
        .head(100)
        .to_string(
            index=False
        )
    )


# ============================================================
# 4. Aitchison zero replacement policy
# ============================================================
#
# Aitchison geometry requires strictly positive compositions.
#
# If exact zeros exist:
#
#   delta = 0.5 * minimum positive share in the frozen data
#
# Each zero becomes delta, and all non-zero shares are
# multiplicatively rescaled so the composition still sums to 1.
#
# We will ALSO test delta/10 and delta*10 later.
#
# If there are no zeros, no replacement occurs.
#
# ============================================================

AITCHISON_DELTA = (
    GLOBAL_MIN_POSITIVE
    *
    0.5
)


print("\nAITCHISON ZERO POLICY")

if EXACT_ZERO_COUNT == 0:

    print(
        "No exact zeros found."
    )

    print(
        "Aitchison analysis will use the "
        "original compositions directly."
    )

else:

    print(
        "Exact zeros found:",
        EXACT_ZERO_COUNT
    )

    print(
        "Primary replacement delta:",
        AITCHISON_DELTA
    )

    print(
        "Sensitivity deltas:",
        AITCHISON_DELTA / 10,
        AITCHISON_DELTA,
        AITCHISON_DELTA * 10,
    )


# ============================================================
# 5. Helper functions
# ============================================================

def close_composition(
    x
):

    x = np.asarray(
        x,
        dtype=float,
    )


    total = x.sum(
        axis=-1,
        keepdims=True,
    )


    if (
        total <= 0
    ).any():

        raise RuntimeError(
            "Composition with non-positive total."
        )


    return (
        x
        /
        total
    )


def multiplicative_zero_replacement(
    matrix,
    delta,
):

    matrix = close_composition(
        matrix
    )


    out = matrix.copy()


    for i in range(
        out.shape[0]
    ):

        row = out[
            i
        ]


        zero = (
            row
            ==
            0
        )


        m = int(
            zero.sum()
        )


        if m == 0:

            continue


        if (
            m
            *
            delta
            >=
            1
        ):

            raise RuntimeError(
                "Zero replacement delta too large."
            )


        positive = ~zero


        positive_sum = row[
            positive
        ].sum()


        row[
            zero
        ] = delta


        row[
            positive
        ] = (
            row[
                positive
            ]
            *
            (
                1
                -
                m
                *
                delta
            )
            /
            positive_sum
        )


        out[
            i
        ] = row


    return out


def clr_transform(
    matrix,
    delta=AITCHISON_DELTA,
):

    matrix = close_composition(
        matrix
    )


    if (
        matrix
        ==
        0
    ).any():

        matrix = (
            multiplicative_zero_replacement(
                matrix,
                delta,
            )
        )


    log_x = np.log(
        matrix
    )


    clr = (
        log_x
        -
        log_x.mean(
            axis=1,
            keepdims=True,
        )
    )


    return clr


def rank_matrix(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )


    return np.vstack(
        [
            rankdata(
                -row,
                method="average",
            )
            for row
            in matrix
        ]
    )


def row_unit_normalize(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float,
    )


    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )


    if (
        norms <= 0
    ).any():

        raise RuntimeError(
            "Zero-norm row."
        )


    return (
        matrix
        /
        norms
    )


def centered_rank_unit(
    ranks
):

    centered = (
        ranks
        -
        ranks.mean(
            axis=1,
            keepdims=True,
        )
    )


    return row_unit_normalize(
        centered
    )


def kendalls_w_from_ranks(
    ranks
):

    ranks = np.asarray(
        ranks,
        dtype=float,
    )


    m, n = ranks.shape


    rank_sums = ranks.sum(
        axis=0
    )


    mean_rank_sum = (
        rank_sums.mean()
    )


    S = np.sum(
        (
            rank_sums
            -
            mean_rank_sum
        ) ** 2
    )


    # --------------------------------------------------------
    # Tie correction
    # --------------------------------------------------------

    tie_total = 0.0


    for row in ranks:

        _, counts = np.unique(
            row,
            return_counts=True,
        )


        tied_counts = counts[
            counts > 1
        ]


        tie_total += np.sum(
            tied_counts ** 3
            -
            tied_counts
        )


    denominator = (
        (
            m ** 2
        )
        *
        (
            n ** 3
            -
            n
        )
        -
        m
        *
        tie_total
    )


    if denominator <= 0:

        return np.nan


    return float(
        12
        *
        S
        /
        denominator
    )


def build_profile_matrix(
    outcome,
    remove_belong=False,
):

    temp = (
        rankings[
            rankings[
                "Outcome"
            ]
            ==
            outcome
        ]
        .copy()
    )


    systems = sorted(
        temp[
            "Country"
        ]
        .unique()
    )


    features = (
        FEATURES_13
        if remove_belong
        else FEATURES_14
    )


    rows = []


    for country in systems:

        system_data = (
            temp[
                temp[
                    "Country"
                ]
                ==
                country
            ]
            .set_index(
                "Feature"
            )
        )


        vector = (
            system_data
            .loc[
                features,
                "ImportanceShare",
            ]
            .to_numpy(
                dtype=float
            )
        )


        vector = (
            vector
            /
            vector.sum()
        )


        rows.append(
            vector
        )


    matrix = np.vstack(
        rows
    )


    return (
        systems,
        features,
        matrix,
    )


# ============================================================
# 6. Observed pairwise metric calculation
# ============================================================

def observed_metrics(
    matrix,
):

    matrix = close_composition(
        matrix
    )


    n = matrix.shape[
        0
    ]


    # --------------------------------------------------------
    # Cosine
    # --------------------------------------------------------

    unit = row_unit_normalize(
        matrix
    )


    cosine_matrix = (
        unit
        @
        unit.T
    )


    # --------------------------------------------------------
    # Hellinger
    # H(p,q) =
    #   ||sqrt(p)-sqrt(q)|| / sqrt(2)
    # --------------------------------------------------------

    sqrt_matrix = np.sqrt(
        matrix
    )


    hellinger_matrix = (
        squareform(
            pdist(
                sqrt_matrix,
                metric="euclidean",
            )
        )
        /
        np.sqrt(
            2
        )
    )


    # --------------------------------------------------------
    # Aitchison
    # Euclidean distance in CLR space
    # --------------------------------------------------------

    clr = clr_transform(
        matrix
    )


    aitchison_matrix = (
        squareform(
            pdist(
                clr,
                metric="euclidean",
            )
        )
    )


    # --------------------------------------------------------
    # Rank agreement
    # --------------------------------------------------------

    ranks = rank_matrix(
        matrix
    )


    rank_unit = (
        centered_rank_unit(
            ranks
        )
    )


    spearman_matrix = (
        rank_unit
        @
        rank_unit.T
    )


    kendall_w = (
        kendalls_w_from_ranks(
            ranks
        )
    )


    # --------------------------------------------------------
    # Upper triangle
    # --------------------------------------------------------

    i, j = np.triu_indices(
        n,
        k=1,
    )


    result = {

        "MeanCosine":
            float(
                cosine_matrix[
                    i,
                    j
                ].mean()
            ),

        "MeanHellinger":
            float(
                hellinger_matrix[
                    i,
                    j
                ].mean()
            ),

        "MeanAitchison":
            float(
                aitchison_matrix[
                    i,
                    j
                ].mean()
            ),

        "MeanSpearman":
            float(
                spearman_matrix[
                    i,
                    j
                ].mean()
            ),

        "KendallW":
            float(
                kendall_w
            ),

        "CosineMatrix":
            cosine_matrix,

        "HellingerMatrix":
            hellinger_matrix,

        "AitchisonMatrix":
            aitchison_matrix,

        "SpearmanMatrix":
            spearman_matrix,

        "Ranks":
            ranks,

        "Unit":
            unit,

        "Sqrt":
            sqrt_matrix,

        "CLR":
            clr,

        "RankUnit":
            rank_unit,
    }


    return result


# ============================================================
# 7. Fast permutation-null engine
# ============================================================

def permutation_null(
    matrix,
    seed,
):

    matrix = close_composition(
        matrix
    )


    n_systems, n_features = (
        matrix.shape
    )


    pair_i, pair_j = (
        np.triu_indices(
            n_systems,
            k=1,
        )
    )


    n_pairs = len(
        pair_i
    )


    # --------------------------------------------------------
    # Pre-transform.
    #
    # Predictor-label permutation simply permutes columns.
    # CLR, sqrt, and ranks are therefore permutation-equivariant.
    # --------------------------------------------------------

    unit_base = (
        row_unit_normalize(
            matrix
        )
    )


    sqrt_base = np.sqrt(
        matrix
    )


    clr_base = clr_transform(
        matrix
    )


    ranks_base = rank_matrix(
        matrix
    )


    rank_unit_base = (
        centered_rank_unit(
            ranks_base
        )
    )


    # --------------------------------------------------------
    # Kendall tie denominator is constant under permutation
    # --------------------------------------------------------

    m = n_systems

    n = n_features


    tie_total = 0.0


    for row in ranks_base:

        _, counts = np.unique(
            row,
            return_counts=True,
        )


        tied = counts[
            counts > 1
        ]


        tie_total += np.sum(
            tied ** 3
            -
            tied
        )


    kendall_denominator = (
        (
            m ** 2
        )
        *
        (
            n ** 3
            -
            n
        )
        -
        m
        *
        tie_total
    )


    # --------------------------------------------------------
    # Storage
    # --------------------------------------------------------

    cosine_null = np.empty(
        N_PERMUTATIONS
    )

    hellinger_null = np.empty(
        N_PERMUTATIONS
    )

    aitchison_null = np.empty(
        N_PERMUTATIONS
    )

    spearman_null = np.empty(
        N_PERMUTATIONS
    )

    kendall_null = np.empty(
        N_PERMUTATIONS
    )


    rng = np.random.default_rng(
        seed
    )


    cursor = 0


    while cursor < N_PERMUTATIONS:

        b = min(
            BATCH_SIZE,
            N_PERMUTATIONS
            -
            cursor,
        )


        # ----------------------------------------------------
        # Independent predictor-label permutation for
        # every system in every null draw
        # ----------------------------------------------------

        random_values = rng.random(
            (
                b,
                n_systems,
                n_features,
            )
        )


        permutation_index = np.argsort(
            random_values,
            axis=2,
        )


        # ----------------------------------------------------
        # COSINE
        # ----------------------------------------------------

        perm_unit = np.take_along_axis(
            np.broadcast_to(
                unit_base,
                (
                    b,
                    n_systems,
                    n_features,
                ),
            ),
            permutation_index,
            axis=2,
        )


        summed = (
            perm_unit.sum(
                axis=1
            )
        )


        cosine_batch = (
            (
                np.sum(
                    summed ** 2,
                    axis=1,
                )
                -
                n_systems
            )
            /
            (
                n_systems
                *
                (
                    n_systems
                    -
                    1
                )
            )
        )


        # ----------------------------------------------------
        # SPEARMAN
        # ----------------------------------------------------

        perm_rank_unit = np.take_along_axis(
            np.broadcast_to(
                rank_unit_base,
                (
                    b,
                    n_systems,
                    n_features,
                ),
            ),
            permutation_index,
            axis=2,
        )


        summed_rank = (
            perm_rank_unit.sum(
                axis=1
            )
        )


        spearman_batch = (
            (
                np.sum(
                    summed_rank ** 2,
                    axis=1,
                )
                -
                n_systems
            )
            /
            (
                n_systems
                *
                (
                    n_systems
                    -
                    1
                )
            )
        )


        # ----------------------------------------------------
        # KENDALL W
        # ----------------------------------------------------

        perm_ranks = np.take_along_axis(
            np.broadcast_to(
                ranks_base,
                (
                    b,
                    n_systems,
                    n_features,
                ),
            ),
            permutation_index,
            axis=2,
        )


        rank_sums = (
            perm_ranks.sum(
                axis=1
            )
        )


        rank_sum_mean = (
            rank_sums.mean(
                axis=1,
                keepdims=True,
            )
        )


        S = np.sum(
            (
                rank_sums
                -
                rank_sum_mean
            ) ** 2,
            axis=1,
        )


        kendall_batch = (
            12
            *
            S
            /
            kendall_denominator
        )


        # ----------------------------------------------------
        # HELLINGER
        # ----------------------------------------------------

        perm_sqrt = np.take_along_axis(
            np.broadcast_to(
                sqrt_base,
                (
                    b,
                    n_systems,
                    n_features,
                ),
            ),
            permutation_index,
            axis=2,
        )


        diff_h = (
            perm_sqrt[
                :,
                pair_i,
                :
            ]
            -
            perm_sqrt[
                :,
                pair_j,
                :
            ]
        )


        pair_h = (
            np.sqrt(
                np.sum(
                    diff_h ** 2,
                    axis=2,
                )
            )
            /
            np.sqrt(
                2
            )
        )


        hellinger_batch = (
            pair_h.mean(
                axis=1
            )
        )


        del diff_h
        del pair_h
        del perm_sqrt


        # ----------------------------------------------------
        # AITCHISON
        # ----------------------------------------------------

        perm_clr = np.take_along_axis(
            np.broadcast_to(
                clr_base,
                (
                    b,
                    n_systems,
                    n_features,
                ),
            ),
            permutation_index,
            axis=2,
        )


        diff_a = (
            perm_clr[
                :,
                pair_i,
                :
            ]
            -
            perm_clr[
                :,
                pair_j,
                :
            ]
        )


        pair_a = np.sqrt(
            np.sum(
                diff_a ** 2,
                axis=2,
            )
        )


        aitchison_batch = (
            pair_a.mean(
                axis=1
            )
        )


        del diff_a
        del pair_a
        del perm_clr


        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        end = (
            cursor
            +
            b
        )


        cosine_null[
            cursor:end
        ] = cosine_batch


        hellinger_null[
            cursor:end
        ] = hellinger_batch


        aitchison_null[
            cursor:end
        ] = aitchison_batch


        spearman_null[
            cursor:end
        ] = spearman_batch


        kendall_null[
            cursor:end
        ] = kendall_batch


        cursor = end


    return {

        "MeanCosine":
            cosine_null,

        "MeanHellinger":
            hellinger_null,

        "MeanAitchison":
            aitchison_null,

        "MeanSpearman":
            spearman_null,

        "KendallW":
            kendall_null,
    }


# ============================================================
# 8. Permutation result summarizer
# ============================================================

HIGHER_MORE_ALIGNED = {
    "MeanCosine",
    "MeanSpearman",
    "KendallW",
}


LOWER_MORE_ALIGNED = {
    "MeanHellinger",
    "MeanAitchison",
}


def summarize_null_metric(
    observed,
    null_values,
    metric,
):

    null_values = np.asarray(
        null_values,
        dtype=float,
    )


    null_mean = float(
        null_values.mean()
    )


    null_sd = float(
        null_values.std(
            ddof=1
        )
    )


    if metric in HIGHER_MORE_ALIGNED:

        p_value = (
            1
            +
            int(
                (
                    null_values
                    >=
                    observed
                )
                .sum()
            )
        ) / (
            len(
                null_values
            )
            +
            1
        )


        direction = (
            "higher = greater alignment"
        )


    elif metric in LOWER_MORE_ALIGNED:

        p_value = (
            1
            +
            int(
                (
                    null_values
                    <=
                    observed
                )
                .sum()
            )
        ) / (
            len(
                null_values
            )
            +
            1
        )


        direction = (
            "lower = greater alignment"
        )


    else:

        raise ValueError(
            metric
        )


    standardized_difference = (
        (
            observed
            -
            null_mean
        )
        /
        null_sd
        if null_sd > 0
        else np.nan
    )


    return {

        "Observed":
            float(
                observed
            ),

        "NullMean":
            null_mean,

        "NullSD":
            null_sd,

        "NullQ001":
            float(
                np.quantile(
                    null_values,
                    0.001,
                )
            ),

        "NullQ025":
            float(
                np.quantile(
                    null_values,
                    0.025,
                )
            ),

        "NullQ500":
            float(
                np.quantile(
                    null_values,
                    0.500,
                )
            ),

        "NullQ975":
            float(
                np.quantile(
                    null_values,
                    0.975,
                )
            ),

        "NullQ999":
            float(
                np.quantile(
                    null_values,
                    0.999,
                )
            ),

        "ObservedMinusNull":
            float(
                observed
                -
                null_mean
            ),

        "StandardizedDifference":
            float(
                standardized_difference
            ),

        "PermutationP":
            float(
                p_value
            ),

        "Direction":
            direction,
    }


# ============================================================
# 9. Main observed + permutation analyses
# ============================================================

summary_rows = []

null_long_rows = []

pairwise_rows = []

distinctiveness_rows = []

metric_agreement_rows = []


analysis_start = time.time()


for outcome_index, outcome in enumerate(
    OUTCOMES
):

    for profile_index, (
        profile_name,
        remove_belong,
    ) in enumerate(
        [
            (
                "Full14",
                False,
            ),
            (
                "NoBELONG13",
                True,
            ),
        ]
    ):

        print("\n")
        print("=" * 115)

        print(
            outcome,
            "—",
            profile_name
        )

        print("=" * 115)


        (
            systems,
            features,
            matrix,
        ) = build_profile_matrix(
            outcome=outcome,
            remove_belong=remove_belong,
        )


        observed = (
            observed_metrics(
                matrix
            )
        )


        print(
            f"N systems: {len(systems)}"
        )

        print(
            f"N features: {len(features)}"
        )

        print(
            "Observed mean cosine:",
            round(
                observed[
                    "MeanCosine"
                ],
                6,
            )
        )

        print(
            "Observed mean Aitchison distance:",
            round(
                observed[
                    "MeanAitchison"
                ],
                6,
            )
        )

        print(
            "Observed mean Hellinger distance:",
            round(
                observed[
                    "MeanHellinger"
                ],
                6,
            )
        )

        print(
            "Observed mean Spearman:",
            round(
                observed[
                    "MeanSpearman"
                ],
                6,
            )
        )

        print(
            "Observed Kendall W:",
            round(
                observed[
                    "KendallW"
                ],
                6,
            )
        )


        # ----------------------------------------------------
        # Save observed pairwise values
        # ----------------------------------------------------

        pair_i, pair_j = (
            np.triu_indices(
                len(
                    systems
                ),
                k=1,
            )
        )


        for i, j in zip(
            pair_i,
            pair_j,
        ):

            pairwise_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Profile":
                        profile_name,

                    "System1":
                        systems[
                            i
                        ],

                    "System2":
                        systems[
                            j
                        ],

                    "Cosine":
                        observed[
                            "CosineMatrix"
                        ][
                            i,
                            j
                        ],

                    "Aitchison":
                        observed[
                            "AitchisonMatrix"
                        ][
                            i,
                            j
                        ],

                    "Hellinger":
                        observed[
                            "HellingerMatrix"
                        ][
                            i,
                            j
                        ],

                    "Spearman":
                        observed[
                            "SpearmanMatrix"
                        ][
                            i,
                            j
                        ],
                }
            )


        # ----------------------------------------------------
        # System distinctiveness under each metric
        # ----------------------------------------------------

        n = len(
            systems
        )


        for idx, system in enumerate(
            systems
        ):

            others = np.arange(
                n
            ) != idx


            distinctiveness_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Profile":
                        profile_name,

                    "Country":
                        system,

                    "CosineDistinctiveness":
                        (
                            1
                            -
                            observed[
                                "CosineMatrix"
                            ][
                                idx,
                                others
                            ]
                            .mean()
                        ),

                    "AitchisonDistinctiveness":
                        observed[
                            "AitchisonMatrix"
                        ][
                            idx,
                            others
                        ]
                        .mean(),

                    "HellingerDistinctiveness":
                        observed[
                            "HellingerMatrix"
                        ][
                            idx,
                            others
                        ]
                        .mean(),

                    "RankDistinctiveness":
                        (
                            1
                            -
                            observed[
                                "SpearmanMatrix"
                            ][
                                idx,
                                others
                            ]
                            .mean()
                        ),
                }
            )


        # ----------------------------------------------------
        # Metric agreement
        #
        # Distances are multiplied by -1 so that larger values
        # always mean MORE similar for this correlation.
        # ----------------------------------------------------

        pair_vectors = {

            "Cosine":
                observed[
                    "CosineMatrix"
                ][
                    pair_i,
                    pair_j
                ],

            "NegativeAitchison":
                -observed[
                    "AitchisonMatrix"
                ][
                    pair_i,
                    pair_j
                ],

            "NegativeHellinger":
                -observed[
                    "HellingerMatrix"
                ][
                    pair_i,
                    pair_j
                ],

            "Spearman":
                observed[
                    "SpearmanMatrix"
                ][
                    pair_i,
                    pair_j
                ],
        }


        metric_names = list(
            pair_vectors.keys()
        )


        for a in range(
            len(
                metric_names
            )
        ):

            for b in range(
                a + 1,
                len(
                    metric_names
                )
            ):

                metric_a = (
                    metric_names[
                        a
                    ]
                )

                metric_b = (
                    metric_names[
                        b
                    ]
                )


                rho, p = spearmanr(
                    pair_vectors[
                        metric_a
                    ],
                    pair_vectors[
                        metric_b
                    ],
                )


                metric_agreement_rows.append(
                    {
                        "Outcome":
                            outcome,

                        "Profile":
                            profile_name,

                        "Metric1":
                            metric_a,

                        "Metric2":
                            metric_b,

                        "PairwiseOrderingSpearman":
                            rho,

                        "P":
                            p,
                    }
                )


        # ----------------------------------------------------
        # Permutation null
        # ----------------------------------------------------

        seed = (
            GLOBAL_RANDOM_SEED
            +
            1000
            *
            outcome_index
            +
            100
            *
            profile_index
        )


        null_result = (
            permutation_null(
                matrix=matrix,
                seed=seed,
            )
        )


        for metric in [
            "MeanCosine",
            "MeanAitchison",
            "MeanHellinger",
            "MeanSpearman",
            "KendallW",
        ]:

            metric_summary = (
                summarize_null_metric(
                    observed=observed[
                        metric
                    ],
                    null_values=null_result[
                        metric
                    ],
                    metric=metric,
                )
            )


            summary_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Profile":
                        profile_name,

                    "N_systems":
                        len(
                            systems
                        ),

                    "N_features":
                        len(
                            features
                        ),

                    "Metric":
                        metric,

                    "Seed":
                        seed,

                    **metric_summary,
                }
            )


            for permutation_index, value in enumerate(
                null_result[
                    metric
                ],
                start=1,
            ):

                null_long_rows.append(
                    {
                        "Outcome":
                            outcome,

                        "Profile":
                            profile_name,

                        "Metric":
                            metric,

                        "Permutation":
                            permutation_index,

                        "Value":
                            value,
                    }
                )


        print("\nPermutation results:")


        for metric in [
            "MeanCosine",
            "MeanAitchison",
            "MeanHellinger",
            "MeanSpearman",
            "KendallW",
        ]:

            temp_summary = (
                summary_rows[
                    -5:
                ]
            )


        print(
            "Permutation null complete."
        )


# ============================================================
# 10. Save main outputs
# ============================================================

similarity_summary = pd.DataFrame(
    summary_rows
)


null_distribution = pd.DataFrame(
    null_long_rows
)


pairwise_metrics = pd.DataFrame(
    pairwise_rows
)


system_distinctiveness = pd.DataFrame(
    distinctiveness_rows
)


metric_agreement = pd.DataFrame(
    metric_agreement_rows
)


similarity_summary.to_csv(
    OUT_DIR
    / "similarity_permutation_summary.csv",
    index=False,
)


null_distribution.to_csv(
    OUT_DIR
    / "similarity_null_distributions.csv",
    index=False,
)


pairwise_metrics.to_csv(
    OUT_DIR
    / "observed_pairwise_metrics.csv",
    index=False,
)


system_distinctiveness.to_csv(
    OUT_DIR
    / "system_distinctiveness_by_metric.csv",
    index=False,
)


metric_agreement.to_csv(
    OUT_DIR
    / "metric_pairwise_ordering_agreement.csv",
    index=False,
)


# ============================================================
# 11. Aitchison zero-replacement sensitivity
# ============================================================

aitchison_sensitivity_rows = []


if EXACT_ZERO_COUNT > 0:

    sensitivity_deltas = {

        "DeltaDiv10":
            AITCHISON_DELTA
            /
            10,

        "PrimaryDelta":
            AITCHISON_DELTA,

        "DeltaTimes10":
            AITCHISON_DELTA
            *
            10,
    }


    for outcome in OUTCOMES:

        for profile_name, remove_belong in [
            (
                "Full14",
                False,
            ),
            (
                "NoBELONG13",
                True,
            ),
        ]:

            (
                systems,
                features,
                matrix,
            ) = build_profile_matrix(
                outcome=outcome,
                remove_belong=remove_belong,
            )


            pair_i, pair_j = (
                np.triu_indices(
                    len(
                        systems
                    ),
                    k=1,
                )
            )


            primary_vector = None


            for delta_name, delta in (
                sensitivity_deltas.items()
            ):

                clr = clr_transform(
                    matrix,
                    delta=delta,
                )


                distance_matrix = (
                    squareform(
                        pdist(
                            clr,
                            metric="euclidean",
                        )
                    )
                )


                vector = (
                    distance_matrix[
                        pair_i,
                        pair_j
                    ]
                )


                if (
                    delta_name
                    ==
                    "PrimaryDelta"
                ):

                    primary_vector = (
                        vector.copy()
                    )


                aitchison_sensitivity_rows.append(
                    {
                        "Outcome":
                            outcome,

                        "Profile":
                            profile_name,

                        "DeltaSetting":
                            delta_name,

                        "Delta":
                            delta,

                        "MeanAitchison":
                            vector.mean(),

                        "MedianAitchison":
                            np.median(
                                vector
                            ),
                    }
                )


            # -----------------------------------------------
            # Compare ordering of pairwise distances with
            # primary replacement choice
            # -----------------------------------------------

            for delta_name, delta in (
                sensitivity_deltas.items()
            ):

                clr = clr_transform(
                    matrix,
                    delta=delta,
                )


                distance_matrix = (
                    squareform(
                        pdist(
                            clr,
                            metric="euclidean",
                        )
                    )
                )


                vector = (
                    distance_matrix[
                        pair_i,
                        pair_j
                    ]
                )


                rho, _ = spearmanr(
                    primary_vector,
                    vector,
                )


                aitchison_sensitivity_rows.append(
                    {
                        "Outcome":
                            outcome,

                        "Profile":
                            profile_name,

                        "DeltaSetting":
                            (
                                delta_name
                                +
                                "_vs_Primary"
                            ),

                        "Delta":
                            delta,

                        "MeanAitchison":
                            np.nan,

                        "MedianAitchison":
                            np.nan,

                        "PairwiseRankCorrelationWithPrimary":
                            rho,
                    }
                )


aitchison_sensitivity = pd.DataFrame(
    aitchison_sensitivity_rows
)


aitchison_sensitivity.to_csv(
    OUT_DIR
    / "aitchison_zero_replacement_sensitivity.csv",
    index=False,
)


# ============================================================
# 12. Compact manuscript-oriented summary
# ============================================================

compact_rows = []


for (
    outcome,
    profile
), group in similarity_summary.groupby(
    [
        "Outcome",
        "Profile",
    ]
):

    row = {
        "Outcome":
            outcome,

        "Profile":
            profile,
    }


    for _, metric_row in group.iterrows():

        metric = (
            metric_row[
                "Metric"
            ]
        )


        row[
            f"{metric}_Observed"
        ] = (
            metric_row[
                "Observed"
            ]
        )


        row[
            f"{metric}_NullMean"
        ] = (
            metric_row[
                "NullMean"
            ]
        )


        row[
            f"{metric}_P"
        ] = (
            metric_row[
                "PermutationP"
            ]
        )


    compact_rows.append(
        row
    )


compact_summary = pd.DataFrame(
    compact_rows
)


compact_summary.to_csv(
    OUT_DIR
    / "similarity_framework_compact.csv",
    index=False,
)


# ============================================================
# 13. Distinctiveness agreement
# ============================================================

distinctiveness_agreement_rows = []


for (
    outcome,
    profile
), group in system_distinctiveness.groupby(
    [
        "Outcome",
        "Profile",
    ]
):

    columns = [
        "CosineDistinctiveness",
        "AitchisonDistinctiveness",
        "HellingerDistinctiveness",
        "RankDistinctiveness",
    ]


    for i in range(
        len(
            columns
        )
    ):

        for j in range(
            i + 1,
            len(
                columns
            )
        ):

            rho, p = spearmanr(
                group[
                    columns[
                        i
                    ]
                ],
                group[
                    columns[
                        j
                    ]
                ],
            )


            distinctiveness_agreement_rows.append(
                {
                    "Outcome":
                        outcome,

                    "Profile":
                        profile,

                    "Metric1":
                        columns[
                            i
                        ],

                    "Metric2":
                        columns[
                            j
                        ],

                    "Spearman":
                        rho,

                    "P":
                        p,
                }
            )


distinctiveness_agreement = pd.DataFrame(
    distinctiveness_agreement_rows
)


distinctiveness_agreement.to_csv(
    OUT_DIR
    / "system_distinctiveness_metric_agreement.csv",
    index=False,
)


# ============================================================
# 14. Print final results
# ============================================================

runtime_minutes = (
    time.time()
    -
    analysis_start
) / 60


print("\n")
print("=" * 120)
print("SIMILARITY FRAMEWORK V2 — COMPLETE")
print("=" * 120)


print(
    f"Runtime: {runtime_minutes:.2f} minutes"
)


print(
    "Exact zero shares:",
    EXACT_ZERO_COUNT
)


print(
    "Minimum positive share:",
    GLOBAL_MIN_POSITIVE
)


print("\n")
print("=" * 120)
print("PERMUTATION NULL SUMMARY")
print("=" * 120)


print(
    similarity_summary[
        [
            "Outcome",
            "Profile",
            "Metric",
            "Observed",
            "NullMean",
            "NullQ025",
            "NullQ975",
            "ObservedMinusNull",
            "StandardizedDifference",
            "PermutationP",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 120)
print("PAIRWISE METRIC AGREEMENT")
print("=" * 120)


print(
    metric_agreement[
        [
            "Outcome",
            "Profile",
            "Metric1",
            "Metric2",
            "PairwiseOrderingSpearman",
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 120)
print("SYSTEM DISTINCTIVENESS METRIC AGREEMENT")
print("=" * 120)


print(
    distinctiveness_agreement[
        [
            "Outcome",
            "Profile",
            "Metric1",
            "Metric2",
            "Spearman",
        ]
    ]
    .round(
        4
    )
    .to_string(
        index=False
    )
)


if EXACT_ZERO_COUNT > 0:

    print("\n")
    print("=" * 120)
    print("AITCHISON ZERO-REPLACEMENT SENSITIVITY")
    print("=" * 120)


    print(
        aitchison_sensitivity
        .round(
            8
        )
        .to_string(
            index=False
        )
    )


print("\n")
print("=" * 120)
print("FILES CREATED")
print("=" * 120)


for path in sorted(
    OUT_DIR.glob(
        "*.csv"
    )
):

    print(
        path
    )



SHAP PROFILE AUDIT
Duplicate Country/Outcome/Feature rows: 0
Invalid/non-finite/negative shares: 0
Profiles: 374
Profiles with !=14 predictors: 0
Profiles with share-sum error > tolerance: 0
Profiles with missing/extra predictors: 0

ZERO / NEAR-ZERO SUMMARY
Exact zero shares: 0
Shares < 1e-8: 0
Shares < 1e-6: 0
Shares < 1e-4: 0
Minimum positive share: 0.0100526145995841

AITCHISON ZERO POLICY
No exact zeros found.
Aitchison analysis will use the original compositions directly.


BEINGBULLIED — Full14
N systems: 74
N features: 14
Observed mean cosine: 0.954609
Observed mean Aitchison distance: 1.360718
Observed mean Hellinger distance: 0.125982
Observed mean Spearman: 0.680701
Observed Kendall W: 0.685016

Permutation results:
Permutation null complete.


BEINGBULLIED — NoBELONG13
N systems: 74
N features: 13
Observed mean cosine: 0.928621
Observed mean Aitchison distance: 1.3087
Observed mean Hellinger distance: 0.132402
Observed mean Spearman: 0.601088
Observed Kendall W: 0.606479


: 